# 🩺 HUẤN LUYỆN & KIỂM CHỨNG HỆ THỐNG CHẨN ĐOÁN BỎNG DA NHẬN THỨC ĐỘ BẤT ĐỊNH KÉP
**Nhánh chuẩn:** `main` · **Dữ liệu chuẩn:** `processed.zip` (MobileSAM Pixel Masks, 1,225 ảnh hợp lệ)

**Cấu trúc thực nghiệm gọn – sạch – chuẩn Q1/Q2:**
1. **Giai đoạn 1 (Phân đoạn & Bất định Biên):** Ablation có kiểm soát từng biến (`ResNet-34 + Combo` vs `Focal Tversky` vs `ConvNeXt + Combo` vs `scSE Attention + Combo`) → **`5-Fold U-Net (ResNet-34) + Combo (+4x TTA)`** + Độ bất định không gian & Khoảng dịch truyền Parkland (Hướng 1).
2. **Giai đoạn 2 (Phân loại & Độ bất định Bằng chứng):** Ablation (`Weighted CE` vs `CB Focal` vs `ConvNeXt CB Focal` vs `EDL softplus` vs `Asymmetric R-EDL exp`) → **`5-Fold Asymmetric R-EDL Ensemble`** + Selective Clinical Triage (`p < 10^-4`).
3. **Kiểm chứng Tổng quát hóa & Phân tích Lỗi (Hướng 4A):** Ngoại viện **`BIP_US` (Tây Ban Nha)**, chặn ảnh bệnh da liễu **`ISIC` OOD**, độ bền nhiễu điện thoại & `Grad-CAM` ca thất bại.

### ⚙️ BƯỚC 1: Kiểm tra GPU Google Colab
*(Khuyến nghị chọn `T4 GPU` khi train; riêng Bước 7 đánh giá có thể chạy được cả trên CPU)*

In [ ]:
!nvidia-smi || echo 'Đang chạy ở chế độ CPU'

### 📁 BƯỚC 2: Kết nối Google Drive & Đồng bộ Mã nguồn (`main`)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

if not os.path.exists('/content/skin-burn-analysis-system'):
    %cd /content
    !git clone -b main https://github.com/Triminh15/skin-burn-analysis-system.git
    %cd /content/skin-burn-analysis-system
else:
    %cd /content/skin-burn-analysis-system
    !git fetch origin main
    !git checkout -f main
    !git pull origin main

!pwd

### 📦 BƯỚC 3: Cài đặt Thư viện Phụ thuộc

In [ ]:
%cd /content/skin-burn-analysis-system
!pip install -q timm albumentations segmentation-models-pytorch matplotlib pandas tqdm scikit-learn scipy pyarrow pillow

### 🗄️ BƯỚC 4: Giải nén Dữ liệu Chuẩn (`processed.zip`)
*(Chỉ dùng `processed.zip` chứa mask MobileSAM thật; tuyệt đối không dùng `data_processed.zip` cũ)*

In [ ]:
%cd /content/skin-burn-analysis-system
import os, zipfile, shutil

candidate_zips = [
    '/content/drive/MyDrive/processed.zip',
    '/content/drive/MyDrive/skin_burn/processed.zip',
    '/content/drive/MyDrive/Do_An_tri/processed.zip',
    '/content/processed.zip',
    'processed.zip',
]

zip_path = next((p for p in candidate_zips if os.path.exists(p)), None)

if not zip_path and os.path.exists('/content/drive/MyDrive'):
    for root, _, files_in_dir in os.walk('/content/drive/MyDrive'):
        if '.Trash' in root or '.shortcut-targets-by-id' in root:
            continue
        if 'processed.zip' in files_in_dir:
            zip_path = os.path.join(root, 'processed.zip')
            break

if not zip_path:
    print('⚠️ Chưa thấy processed.zip trên Drive — chọn file processed.zip từ máy tính để upload:')
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            zip_path = fn
            if os.path.exists('/content/drive/MyDrive'):
                shutil.copy(fn, os.path.join('/content/drive/MyDrive', 'processed.zip'))
            break

assert zip_path and os.path.exists(zip_path), '❌ Không tìm thấy processed.zip!'
print(f'📦 Đang giải nén {zip_path}...')
os.makedirs('data/processed', exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    for member in zf.infolist():
        clean_name = member.filename.replace('\\', '/').strip('/')
        if member.is_dir() or clean_name.endswith('/'):
            continue
        if 'images/' in clean_name:
            sub_rel = 'images/' + clean_name.split('images/')[-1]
        elif 'masks/' in clean_name:
            sub_rel = 'masks/' + clean_name.split('masks/')[-1]
        elif 'rois/' in clean_name:
            sub_rel = 'rois/' + clean_name.split('rois/')[-1]
        else:
            sub_rel = clean_name
        dest_path = os.path.join('data/processed', sub_rel)
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        with zf.open(member) as src, open(dest_path, 'wb') as dst:
            dst.write(src.read())

print(f"✅ Images: {len(os.listdir('data/processed/images'))} | Masks: {len(os.listdir('data/processed/masks'))} | ROIs: {len(os.listdir('data/processed/rois'))}")

## 🧪 BƯỚC 5: ABLATION STUDY CÓ KIỂM SOÁT (1-MODEL, CỐ ĐỊNH `seed=42`)
### 🔹 5A. Giai đoạn 1 — Phân đoạn (Controlled One-Factor-at-a-Time Ablation)
1. `U-Net (ResNet-34) + ComboLoss` *(Champion)*
2. `U-Net (ResNet-34) + Focal Tversky` *(Đổi biến Loss)*
3. `U-Net (ConvNeXt-Tiny) + ComboLoss` *(Đổi biến Backbone)*
4. `Att U-Net (ResNet-34 + scSE) + ComboLoss` *(Đổi biến Attention)*

In [ ]:
%cd /content/skin-burn-analysis-system
!python src/vision/train_seg.py --arch unet --backbone resnet34 --loss combo --epochs 35 --batch-size 16 --run-name unet_resnet34_combo
!python src/vision/train_seg.py --arch unet --backbone resnet34 --loss focal_tversky --epochs 35 --batch-size 16 --run-name unet_resnet34_focal_tversky
!python src/vision/train_seg.py --arch unet --backbone tu-convnext_tiny --loss combo --epochs 35 --batch-size 8 --run-name unet_convnext_combo
!python src/vision/train_seg.py --arch unet --backbone resnet34 --attention scse --loss combo --epochs 35 --batch-size 16 --run-name att_unet_resnet34_combo

# Tự động sao lưu lên Drive ngay sau khi xong
!mkdir -p /content/drive/MyDrive/burn_research_outputs/checkpoints /content/drive/MyDrive/burn_research_outputs/figures
!cp -ru outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/checkpoints/
!cp -ru outputs/figures/* /content/drive/MyDrive/burn_research_outputs/figures/

### 🔹 5B. Giai đoạn 2 — Phân loại & Độ bất định (`ResNet-34` / `ConvNeXt` / `EDL softplus` / `Asymmetric R-EDL exp`)

In [ ]:
%cd /content/skin-burn-analysis-system
!python src/vision/train_cls.py --model resnet34 --loss weighted_ce --epochs 30 --batch-size 16 --output-dir outputs/checkpoints/classifier --run-name cls_resnet34_weighted_ce
!python src/vision/train_cls.py --model resnet34 --loss cb_focal --epochs 30 --batch-size 16 --output-dir outputs/checkpoints/classifier --run-name cls_resnet34_cb_focal
!python src/vision/train_cls.py --model convnext_tiny --loss cb_focal --epochs 30 --batch-size 16 --output-dir outputs/checkpoints/classifier --run-name cls_convnext_cb_focal
!python src/vision/train_cls.py --model resnet34 --loss edl --edl-activation softplus --kl-weight 0.04 --deg3-boost 1.0 --epochs 30 --batch-size 16 --output-dir outputs/checkpoints/classifier --run-name cls_resnet34_edl_softplus
!python src/vision/train_cls.py --model resnet34 --loss edl --edl-type digamma --edl-activation exp --kl-weight 0.04 --deg3-boost 1.35 --epochs 30 --batch-size 16 --lr 3e-4 --annealing-epochs 10 --output-dir outputs/checkpoints/classifier --run-name run_m1_classifier_redl_exp

!mkdir -p /content/drive/MyDrive/burn_research_outputs/checkpoints /content/drive/MyDrive/burn_research_outputs/figures
!cp -ru outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/checkpoints/
!cp -ru outputs/figures/* /content/drive/MyDrive/burn_research_outputs/figures/

## 🏅 BƯỚC 6: SYNCHRONIZED STRATIFIED 5-FOLD CROSS-VALIDATION & SELF-ENSEMBLE (CHUẨN Q1/Q2)
- Khóa cứng 187 ảnh Test độc lập, đồng bộ `image_id` giữa 2 giai đoạn.
- Chạy 5-Fold cho cấu hình vô địch Phân đoạn (`U-Net ResNet-34 + Combo`) và đối chứng Phân loại (`Asymmetric R-EDL` vs `Weighted CE`).

In [ ]:
# 6.1) 5-Fold Phân đoạn (U-Net ResNet-34 + Combo + 4x TTA)
%cd /content/skin-burn-analysis-system
!python tools/run_5fold_cv.py --task seg --seg-arch unet --seg-backbone resnet34 --seg-loss combo --seg-epochs 35 --patience 10

!mkdir -p /content/drive/MyDrive/burn_research_outputs/checkpoints /content/drive/MyDrive/burn_research_outputs/figures
!cp -ru outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/checkpoints/
!cp -ru outputs/figures/* /content/drive/MyDrive/burn_research_outputs/figures/

In [ ]:
# 6.2) 5-Fold Phân loại chính (Asymmetric R-EDL + OOF Bayes + Selective Prediction)
%cd /content/skin-burn-analysis-system
!python tools/run_5fold_cv.py --task cls --cls-model resnet34 --cls-loss edl --edl-activation exp --kl-weight 0.04 --deg3-boost 1.0 --cls-epochs 30 --cls-patience 8

# 6.3) 5-Fold Phân loại đối chứng công bằng (Weighted CE)
!python tools/run_5fold_cv.py --task cls --cls-model resnet34 --cls-loss weighted_ce --cls-epochs 30 --cls-patience 8

!mkdir -p /content/drive/MyDrive/burn_research_outputs/checkpoints /content/drive/MyDrive/burn_research_outputs/figures
!cp -ru outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/checkpoints/
!cp -ru outputs/figures/* /content/drive/MyDrive/burn_research_outputs/figures/

## 🛡️ BƯỚC 7: ĐÁNH GIÁ BẤT ĐỊNH KÉP (HƯỚNG 1) & NGOẠI VIỆN `BIP_US` / `ISIC OOD` + PHÂN TÍCH LỖI (HƯỚNG 4A)
- **0 phút train lại!** Tự động nạp trực tiếp checkpoints từ `outputs/checkpoints/` hoặc từ Google Drive.
- Xuất trọn bộ 4 biểu đồ 300 DPI + báo cáo JSON/CSV vào `outputs/figures/`.

In [ ]:
%cd /content/skin-burn-analysis-system
!python tools/download_external_datasets.py
!python tools/evaluate_spatial_uncertainty.py --arch unet --backbone resnet34 --loss combo
!python tools/evaluate_ood_and_generalization.py

### 💾 BƯỚC 8: Sao lưu Toàn bộ Checkpoints, JSON & Biểu đồ 300 DPI vào Google Drive

In [ ]:
%cd /content/skin-burn-analysis-system
!mkdir -p /content/drive/MyDrive/burn_research_outputs/checkpoints /content/drive/MyDrive/burn_research_outputs/figures
!cp -ru outputs/checkpoints/* /content/drive/MyDrive/burn_research_outputs/checkpoints/
!cp -ru outputs/figures/* /content/drive/MyDrive/burn_research_outputs/figures/
print('✅ Toàn bộ mô hình, JSON và biểu đồ Q1/Q2 đã được đồng bộ vào /content/drive/MyDrive/burn_research_outputs/')